In [ ]:
#Install GRoq library
!pip install groq -q
print("Libraries installed successfully")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 6.3 MB/s eta 0:00:00
Libraries installed successfully


**Import all Required libraries**

In [ ]:

import sqlite3
import pandas as pd
import os
from groq import Groq
import re
print("All libraries imported successfully")

All libraries imported successfully


In [ ]:
import os
os.environ["GROQ_API_KEY"] = "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"
client = Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL = "llama-3.1-8b-instant"
print("Groq client initialized successfully")
print(f"Using model:{MODEL}")

Groq client initialized successfully
Using model:llama-3.1-8b-instant


In [ ]:
#In a real session ,students upload the provided CSV file
import io
csv_data = """student_id,name,age,gender,subject,marks,attendance,grade
1,Aarav Sharma,20,Male,Mathematics,88,92,A
2,Priya Patel,21,Female,Science,76,85,B
3,Rohan Mehta,20,Male,Programming,95,98,A+
4,Sneha Iyer,22,Female,Mathematics,62,78,C
5,Arjun Nair,21,Male,Programming,91,94,A+
6,Divya Krishnan,20,Female,Science,83,88,A
7,Karan Singh,22,Male,Mathematics,74,81,B
8,Ananya Gupta,21,Female,Programming,89,96,A
9,Vikram Reddy,20,Male,Science,70,79,B
10,Pooja Sharma,22,Female,Mathematics,55,72,D
11,Aditya Kumar,21,Male,Programming,97,99,A+
12,Meera Nambiar,20,Female,Science,81,87,A
13,Rahul Desai,22,Male,Mathematics,68,80,C
14,Kavitha Rajan,21,Female,Programming,86,93,A
15,Nikhil Verma,20,Male,Science,77,84,B
16,Swathi Pillai,22,Female,Mathematics,90,95,A+
17,Manish Joshi,21,Male,Programming,73,82,B
18,Lavanya Menon,20,Female,Science,66,76,C
19,Suresh Babu,22,Male,Mathematics,82,89,A
20,Anjali Singh,21,Female,Programming,94,97,A+
21,Deepak Nair,20,Male,Science,79,86,B
22,Rekha Sharma,22,Female,Mathematics,58,73,D
23,Sanjay Patel,21,Male,Programming,88,91,A
24,Usha Iyer,20,Female,Science,84,90,A
25,Vijay Kumar,22,Male,Mathematics,71,83,B
26,Nandita Rao,21,Female,Programming,92,96,A+
27,Ashok Reddy,20,Male,Science,65,77,C
28,Sunita Gupta,22,Female,Mathematics,87,93,A
29,Ravi Krishnan,21,Male,Programming,78,88,B
30,Bhavna Mehta,20,Female,Science,93,98,A+"""
#Read the CSV text into a DataFrame
df=pd.read_csv(io.StringIO(csv_data))
print(f"Dataset loaded:{len(df)} rows.{len(df.columns)}columns")
print("\n First 5 rows")
df.head()

Dataset loaded:30 rows.8columns

 First 5 rows


,student_id,name,age,gender,subject,marks,attendance,grade
0,1,Aarav Sharma,20,Male,Mathematics,88,92,A
1,2,Priya Patel,21,Female,Science,76,85,B
2,3,Rohan Mehta,20,Male,Programming,95,98,A+
3,4,Sneha Iyer,22,Female,Mathematics,62,78,C
4,5,Arjun Nair,21,Male,Programming,91,94,A+


In [ ]:
#create a SQLite database and load the data
conn = sqlite3.connect("college.db")
df.to_sql("students",conn,if_exists="replace",index=False)
print("Database created: college.db")
print("Table'students' created with 30 student records")
test_df = pd.read_sql_query("SELECT COUNT(*) as total_rows FROM students",conn)
print(f"\nVerification:{test_df['total_rows'][0]} rows in database")

Database created: college.db
Table'students' created with 30 student records

Verification:30 rows in database


In [ ]:
def get_schema(conn,table_name="students"):
  """
  This function reads the structured of a database table.
  It returns information about each column:name and data types

  Parameters:
  conn;The Sqlite connection object (our link to the database)
  table_name:The name of the table to inspect(default:'stuents)
  Returns:
  A formatted strinf describleing the table structure
"""
  cursor=conn.cursor()
  cursor.execute(f"PRAGMA table_info({table_name})")
  columns=cursor.fetchall()
  schema_lines=[f"Table:{table_name}"]
  schema_lines.append("Columns")
  for col in columns:
    schema_lines.append(f"  -{col[1]} {col[2]}")
  cursor.execute(f"Select * FROM {table_name} LIMIT 3")
  sample_rows=cursor.fetchall()
  schema_lines.append("\n Sample rows (first 3)")
  for row in sample_rows:
    schema_lines.append(f" {row}")
  return "\n" .join(schema_lines)
schema=get_schema(conn)
print(schema)

Table:students
Columns
  -student_id INTEGER
  -name TEXT
  -age INTEGER
  -gender TEXT
  -subject TEXT
  -marks INTEGER
  -attendance INTEGER
  -grade TEXT

 Sample rows (first 3)
 (1, 'Aarav Sharma', 20, 'Male', 'Mathematics', 88, 92, 'A')
 (2, 'Priya Patel', 21, 'Female', 'Science', 76, 85, 'B')
 (3, 'Rohan Mehta', 20, 'Male', 'Programming', 95, 98, 'A+')


In [ ]:
def generate_sql(user_question,schema_text,client,model):
  """Sends the user's question and database schema to the Groq API
  The LLM generates a SQL query that answers the question.

  Parameters:
  user_question:The Question typed by the user(plain English)
  schema_text:The database structure (so the LLM knows column name)
  client:The GRoq client object(Created earlier)
  model:The model name string
  Returns:
  A string containg the SQL query
  """
  system_prompt=f"""You are an expert SQL assisstant.
  You are connected to a SQLlite database with the following structure
  {schema_text}

  Rules you must follow:
  1.Generate ONLY a valid SQLite SQL query
  2.DO not include any explantion or text -only the SQL Query
  3.Do not use markdown code blocks.Return the row SQL only.
  4.The table name  is:students
  5.ONly use column names that exist in the schema above.
  6.Use single quotes for string vales in WHERE clauses(example: WHERE subject='Programming)
  7.If the user asks for top N,use ORDER BY marks DESC LIMIT N
  """
  response=client.chat.completions.create(
      model=model,
      messages=[
          {"role":"system","content":system_prompt},
          {"role":"user","content":user_question}
      ],
      temperature=0.0


  )

  sql_query=response.choices[0].message.content.strip()
  return sql_query
question="Show me all female students"
print(f"Question:{question}")
print("\n Genrating SQL...")
sql=generate_sql(question,schema,client,MODEL)

print(f"\n Generated SQL:\n{sql}")

Question:Show me all female students

 Genrating SQL...

 Generated SQL:
SELECT * FROM students WHERE gender='Female'


In [17]:
def execute_sql(sql_query, conn):
  """
  Cleans the AI-generated SQL and executes it on the SQLite database
  Returns the results as a pandas DataFrame.
  Parameters:
  """
  clean_sql = sql_query
  clean_sql = re.sub(r'```sql\s*', '', clean_sql)
  clean_sql = re.sub(r'\s*```', '', clean_sql)
  clean_sql = clean_sql.strip()
  try:
    result_df = pd.read_sql_query(clean_sql, conn)
    return result_df, None
  except Exception as e:

    return None,str(e)

print(f"Executing SQL:{sql}")
result,error=execute_sql(sql,conn)
if error:
  print(f"Error:{error}")
else:
  print(f"\n Query returned {len(result)} rows")
  print(result)

Executing SQL:SELECT * FROM students WHERE gender='Female'

 Query returned 15 rows
    student_id            name  age  gender      subject  marks  attendance  \
0            2     Priya Patel   21  Female      Science     76          85   
1            4      Sneha Iyer   22  Female  Mathematics     62          78   
2            6  Divya Krishnan   20  Female      Science     83          88   
3            8    Ananya Gupta   21  Female  Programming     89          96   
4           10    Pooja Sharma   22  Female  Mathematics     55          72   
5           12   Meera Nambiar   20  Female      Science     81          87   
6           14   Kavitha Rajan   21  Female  Programming     86          93   
7           16   Swathi Pillai   22  Female  Mathematics     90          95   
8           18   Lavanya Menon   20  Female      Science     66          76   
9           20    Anjali Singh   21  Female  Programming     94          97   
10          22    Rekha Sharma   22  Female  Ma

In [20]:
# the Complete Text-to_sql Agent
def text_to_sql_agent(user_question,conn,client,model,verbose=True):
  """
  The main AI Agent function
  Takes a user question in plain English and returns database results
  This is the complete workflow:
  1.Get database schema
  2.Generate SQL using Groq LLM
  Parameters:
   user_question: The question typed by the user(plain English)
   conn:Groq API client
   model: Model name string
   verbose: If true,prints each stop(usefull for learning and debugging)
  Returns:
  A tuple:(result_dataframe,generated_sql_string)
  """
  print("=" * 60)
  print(f"USER QUESTION: {user_question}")
  print("=" * 60)
  if verbose:
    print("\n[STEP 1] Reading database Schema...")
  schema_text = get_schema(conn)
  if verbose:
    print("Schema loaded successfully")
  if verbose:
    print("\n[STEP 2] Generating SQL query with Groq LLM...")
  generated_sql = generate_sql(user_question, schema_text,client,model)
  if verbose:
    print("\n[STEP 3] Executing SQL on the database...")
  result_df, error = execute_sql(generated_sql,conn)
  if error:
    print(f"SQL Execution Error:{error}")
    return None,generated_sql
  if verbose:
    print(f"\n[STEP 4] Query returned {len(result_df)} row(s)")
    print("\nRESULTS:")
    print("-" * 40)
    print(result_df.to_string(index=False))
    print("=" * 60)
    return result_df,generated_sql

result,sql_used = text_to_sql_agent(
     "Show top 5 students in Programming",
     conn,client,MODEL

 )

USER QUESTION: Show top 5 students in Programming

[STEP 1] Reading database Schema...
Schema loaded successfully

[STEP 2] Generating SQL query with Groq LLM...

[STEP 3] Executing SQL on the database...

[STEP 4] Query returned 5 row(s)

RESULTS:
----------------------------------------
 student_id         name  age gender     subject  marks  attendance grade
         11 Aditya Kumar   21   Male Programming     97          99    A+
          3  Rohan Mehta   20   Male Programming     95          98    A+
         20 Anjali Singh   21 Female Programming     94          97    A+
         26  Nandita Rao   21 Female Programming     92          96    A+
          5   Arjun Nair   21   Male Programming     91          94    A+


In [22]:
result1=text_to_sql_agent(
    "Show me all students who study Mathematics",
    conn,client,MODEL
)

USER QUESTION: Show me all students who study Mathematics

[STEP 1] Reading database Schema...
Schema loaded successfully

[STEP 2] Generating SQL query with Groq LLM...

[STEP 3] Executing SQL on the database...

[STEP 4] Query returned 10 row(s)

RESULTS:
----------------------------------------
 student_id          name  age gender     subject  marks  attendance grade
          1  Aarav Sharma   20   Male Mathematics     88          92     A
          4    Sneha Iyer   22 Female Mathematics     62          78     C
          7   Karan Singh   22   Male Mathematics     74          81     B
         10  Pooja Sharma   22 Female Mathematics     55          72     D
         13   Rahul Desai   22   Male Mathematics     68          80     C
         16 Swathi Pillai   22 Female Mathematics     90          95    A+
         19   Suresh Babu   22   Male Mathematics     82          89     A
         22  Rekha Sharma   22 Female Mathematics     58          73     D
         25   Vijay Kumar 

In [24]:
#test 2:Aggregation (average)
result2,_=text_to_sql_agent(
    "What is the average marks for each subjects?",
    conn,client,MODEL
)

USER QUESTION: What is the average marks for each subjects?

[STEP 1] Reading database Schema...
Schema loaded successfully

[STEP 2] Generating SQL query with Groq LLM...

[STEP 3] Executing SQL on the database...

[STEP 4] Query returned 3 row(s)

RESULTS:
----------------------------------------
    subject  AVG(marks)
Mathematics        73.5
Programming        88.3
    Science        77.4


In [25]:
#Test 3:Conditional filtering
results3,_=text_to_sql_agent(
    "Show students who scored more than 90",
    conn,client,MODEL
)


USER QUESTION: Show students who scored more than 90

[STEP 1] Reading database Schema...
Schema loaded successfully

[STEP 2] Generating SQL query with Groq LLM...

[STEP 3] Executing SQL on the database...

[STEP 4] Query returned 6 row(s)

RESULTS:
----------------------------------------
 student_id         name  age gender     subject  marks  attendance grade
          3  Rohan Mehta   20   Male Programming     95          98    A+
          5   Arjun Nair   21   Male Programming     91          94    A+
         11 Aditya Kumar   21   Male Programming     97          99    A+
         20 Anjali Singh   21 Female Programming     94          97    A+
         26  Nandita Rao   21 Female Programming     92          96    A+
         30 Bhavna Mehta   20 Female     Science     93          98    A+


In [26]:
#Test 4:Counting
result4,_=text_to_sql_agent(
    "How many male and female students are there?",
    conn,client,MODEL

)

USER QUESTION: How many male and female students are there?

[STEP 1] Reading database Schema...
Schema loaded successfully

[STEP 2] Generating SQL query with Groq LLM...

[STEP 3] Executing SQL on the database...

[STEP 4] Query returned 1 row(s)

RESULTS:
----------------------------------------
 male_count  female_count
         15            15


In [28]:
#Test 5:Complex query-multiple conditions
results5,_=text_to_sql_agent(
    "Show female students who score above 85 in Science or Programming ,ordered by marks",
    conn,client,MODEL


)

USER QUESTION: Show female students who score above 85 in Science or Programming ,ordered by marks

[STEP 1] Reading database Schema...
Schema loaded successfully

[STEP 2] Generating SQL query with Groq LLM...

[STEP 3] Executing SQL on the database...

[STEP 4] Query returned 5 row(s)

RESULTS:
----------------------------------------
 student_id          name  age gender     subject  marks  attendance grade
         20  Anjali Singh   21 Female Programming     94          97    A+
         30  Bhavna Mehta   20 Female     Science     93          98    A+
         26   Nandita Rao   21 Female Programming     92          96    A+
          8  Ananya Gupta   21 Female Programming     89          96     A
         14 Kavitha Rajan   21 Female Programming     86          93     A


In [30]:
def generate_answer(user_question,query_results_df,client,model):
  if query_results_df is None or len(query_results_df)==0:
    return "No results were found or your query."
  results_text=query_results_df.to_string(index=False)
  prompt=f"""The user asked:'{user_question}'
  The database returned these results:
  {results_text}

  Please write a clear,friendly ,2-3 sentence answer to the user's question based on these results.
  Be specific.Mention actual names and numbers from the data.
  Do not add information not present in the results."""
  response=client.chat.completions.create(
      model=model,
      messages=[{'role':"user","content":prompt}],
      temperature=0.3
  )
  return response.choices[0].message.content.strip()

def smart_text_to_sql_agent(user_question,conn,client,model):
  """
  Enhanced agent that returns both table AND a natural language answer.
  """
  print("="*60)
  print(f"Question:{user_question}")
  print("="*60)
  #Step 1:Get Schema
  schema_text=get_schema(conn)
  #Step 2:Generate SQL
  print("Generating SQL...")
  generated_sql=generate_sql(user_question,schema_text,client,model)
  print(f"SQL:{generated_sql}")
  result_df,error=execute_sql(generated_sql,conn)
  if error:
    print(f"Error executing SQL:{error}")
    return
  #Step 4:Show data table
  print(f"\n Data({len(result_df)})rows returned..")
  display(result_df)
  #Step 5:Generated natural language answer
  print("\n Generate natural language answer..")
  answer=generate_answer(user_question,result_df,client,model)
  print("\nAnswer")
  print(answer)
  print("="*60)

#Test the enhanced agent
smart_text_to_sql_agent(
    "Who are the top 5 students in Programming?",
    conn,client,MODEL
)

Question:Who are the top 5 students in Programming?
Generating SQL...
SQL:SELECT name FROM students WHERE subject='Programming' ORDER BY marks DESC LIMIT 5

 Data(5)rows returned..


,name
0,Aditya Kumar
1,Rohan Mehta
2,Anjali Singh
3,Nandita Rao
4,Arjun Nair



 Generate natural language answer..

Answer
Based on the results, the top 5 students in Programming are:

1. Aditya Kumar
2. Rohan Mehta
3. Anjali Singh
4. Nandita Rao
5. Arjun Nair

These students are at the top of the list, indicating their strong performance in the programming course.
